In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "AdditionalVariables")
dataType = "DensityPotentialTemperature"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Setup

Region = "TRACER"; Case = "WET"; spinup_hours = "0"
# Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"
# Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
# Region = "PRECIP"; Case = "DIURNAL"; spinup_hours = "12"

# Region = "Hawaii"; Case = "WET"; spinup_hours = "12"
# Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
###############
#JOB ARRAY SETUP

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_JobArray import JobArray_Class

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs():
    num_jobs=20
    return num_jobs

num_jobs = GetNumJobs()
JobArray = JobArray_Class(total_elements=ModelData_NSSL.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = JobArray.start_job; end_job = JobArray.end_job

def GetNumElements():
    num_elements = np.arange(ModelData_NSSL.Ntime)[start_job:end_job].tolist()
    return num_elements
num_elements = GetNumElements()

In [ ]:
####################################
#FUNCTIONS

In [ ]:
def DensityPotentialTemperature(ModelData, t):
    """
    Calculating Density Potential Temperature (Virtual Potential Temperature with Included Liquid Water.
    For viewing "cold pools" ("outflow boundaries")
    """
    #data
    data = ModelData.GetDataTimestep(t,printout=False).isel(nVertLevels=0) #15 meters
    
    #variables
    Theta = data.theta
    Qv = data.qv
    Ql = data.qc+data.qr
    Qt = Qv+Ql
    
    #constants 
    Rd = 287.04; Rv = 461.5
    epsilon = Rd/Rv
    
    #calculation
    numerator = 1 + Qv/epsilon
    denominator = 1 + Qt
    Theta_rho = Theta*(numerator/denominator)

    variableName = "Theta_rho"
    return Theta_rho,variableName

# def LiquidCorrection(Theta_rho):
#     """
#     Extra: If using the full equation with water correction, however it is usually completely negligible.
#     """
    
#     #constants
#     Alpha_l = 1/1e3
#     Cp = 1005.7; kappa = Rd/Cp
    
#     #variables
#     Pressure = data.pressure
#     T = Theta*((Pressure/1e5)**kappa)
    
#     Pressure_d = (epsilon*Pressure)/(epsilon+Qv)
#     Pressure_v = Pressure - Pressure_d
#     Alpha_d = Rd*T/Pressure_d
    
#     #calculation
#     numerator2 = (1+Ql*(Alpha_l/Alpha_d))
#     Theta_rho2 = Theta_rho*numerator2
#     return Theta_rho2

In [ ]:
def SaveData(ModelData, outputDirectory, outputData, variableName,t,
             printout=True):
    
    outputFolder = f"{ModelData.region}_{ModelData.case}_{ModelData.mpType}_spinup{ModelData.spinup_hours}hrs"
    outputFolderPath = os.path.join(outputDirectory,outputFolder,variableName)
    os.makedirs(outputFolderPath, exist_ok=True)
    
    outputFile = f"{variableName}_{ModelData.timeStrings[t]}.nc"
    outputFilePath = os.path.join(outputFolderPath,outputFile)
        
    outputData.to_netcdf(outputFilePath)

    if printout == True:
        print(f"Outputted to {outputFilePath}","\n")

In [ ]:
####################################
#RUNNING

In [ ]:
for t in tqdm(num_elements, desc="Processing timesteps"):
    for ModelData in [ModelData_NSSL, ModelData_TEMPO]: 
        outputData,variableName = DensityPotentialTemperature(ModelData, t)
        SaveData(ModelData, outputDirectory, outputData, variableName,t)